# KLIFS structure filtering by UniProt

This notebook pulls kinase structures from the KLIFS API and keeps only structures that satisfy:
- resolution <= 2.5
- missing_residues <= 5
- quality_score >= 8.0 (KLIFS field corresponding to quality)

If multiple structures map to the same UniProt accession, the best structure is selected by:
1. highest quality score
2. lowest resolution
3. least missing residues

In [7]:
import requests
import pandas as pd
from typing import Iterable, List

BASE_URL = "https://klifs.net/api"
OUTPUT_CSV = "klifs_filtered_structures_uniprot.csv"
REQUEST_TIMEOUT = 60
KINASE_CHUNK_SIZE = 75

session = requests.Session()
session.headers.update({"Accept": "application/json"})

In [8]:
def chunked(values: List[int], size: int) -> Iterable[List[int]]:
    for i in range(0, len(values), size):
        yield values[i:i + size]


def get_json(endpoint: str, params: dict | None = None):
    response = session.get(f"{BASE_URL}{endpoint}", params=params, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    return response.json()


def fetch_human_kinase_information() -> pd.DataFrame:
    kinase_info = get_json("/kinase_information", params={"species": "HUMAN"})
    kinase_df = pd.DataFrame(kinase_info)

    required = {"kinase_ID", "uniprot"}
    missing = required - set(kinase_df.columns)
    if missing:
        raise ValueError(f"Missing required columns in kinase_information response: {missing}")

    kinase_df = kinase_df[["kinase_ID", "name", "species", "uniprot"]].copy()
    kinase_df["kinase_ID"] = pd.to_numeric(kinase_df["kinase_ID"], errors="coerce").astype("Int64")
    return kinase_df.dropna(subset=["kinase_ID"])


def fetch_structures_for_kinases(kinase_ids: List[int]) -> pd.DataFrame:
    all_structures = []

    for ids in chunked(kinase_ids, KINASE_CHUNK_SIZE):
        params = {"kinase_ID": ",".join(map(str, ids))}
        batch = get_json("/structures_list", params=params)
        if isinstance(batch, list):
            all_structures.extend(batch)

    return pd.DataFrame(all_structures)

In [9]:
# Fetch kinase info
kinase_df = fetch_human_kinase_information()

# Fetch structures for those kinase IDs
kinase_ids = kinase_df["kinase_ID"].dropna().astype(int).unique().tolist()
structures_df = fetch_structures_for_kinases(kinase_ids)

wanted_cols = [
    "structure_ID",
    "kinase_ID",
    "kinase",
    "species",
    "pdb",
    "chain",
    "resolution",
    "quality_score",
    "missing_residues",
]
present_cols = [c for c in wanted_cols if c in structures_df.columns]
structures_df = structures_df[present_cols].copy()

# Merge UniProt onto structures
merged = structures_df.merge(
    kinase_df[["kinase_ID", "uniprot"]],
    on="kinase_ID",
    how="left"
)

# Ensure numeric columns are properly typed
for col in ["resolution", "quality_score", "missing_residues"]:
    if col in merged.columns:
        merged[col] = pd.to_numeric(merged[col], errors="coerce")

# Filters
filtered = merged.loc[
    (merged["resolution"] <= 2.5)
    & (merged["missing_residues"] <= 5)
    & (merged["quality_score"] >= 8.0)
    & (merged["uniprot"].notna())
    & (merged["uniprot"].astype(str).str.strip() != "")
].copy()

# Force unique UniProt
best_per_uniprot = (
    filtered
    .sort_values(
        by=["uniprot", "quality_score", "resolution", "missing_residues"],
        ascending=[True, False, True, True],
        kind="mergesort"
    )
    .drop_duplicates(subset=["uniprot"], keep="first")
    .sort_values("uniprot")
    .reset_index(drop=True)
)

best_per_uniprot.to_csv(OUTPUT_CSV, index=False)

print(f"Saved {len(best_per_uniprot):,} structures to {OUTPUT_CSV}")
best_per_uniprot.head(10)

Saved 225 structures to klifs_filtered_structures_uniprot.csv


,structure_ID,kinase_ID,kinase,species,pdb,chain,resolution,quality_score,missing_residues,uniprot
0,1500,520,BMPR1B,Human,3mdy,C,2.05,9.6,0,O00238
1,13841,267,CDC7,Human,6ya6,A,1.44,8.0,0,O00311
2,1818,314,PLK4,Human,3cok,A,2.25,8.8,0,O00444
3,463,381,YSK1,Human,2xik,A,1.97,9.2,0,O00506
4,12695,389,MAP2K7,Human,6yg0,A,2.00,9.6,0,O14733
5,13496,121,CHK1,Human,7ako,B,1.80,9.9,0,O14757
6,13583,139,CASK,Human,7oai,A,2.30,9.7,0,O14936
7,1977,259,AurA,Human,1mq4,A,1.90,9.6,0,O14965
8,336,279,GAK,Human,4c58,A,2.16,9.4,0,O14976
9,6987,145,DCLK1,Human,5jzj,A,1.71,9.4,0,O15075


In [10]:
vals = (
    best_per_uniprot["uniprot"]
    .dropna()
    .astype(str)
    .str.strip()
)
vals = sorted(vals[vals != ""].drop_duplicates().tolist())

# Comma-separated quoted values
print(", ".join(f"'{v}'" for v in vals))

# Optional: full Python list literal
print("[" + ", ".join(f"'{v}'" for v in vals) + "]")

'O00238', 'O00311', 'O00444', 'O00506', 'O14733', 'O14757', 'O14936', 'O14965', 'O14976', 'O15075', 'O15264', 'O15530', 'O43293', 'O43318', 'O43353', 'O43781', 'O60674', 'O75116', 'O75385', 'O75460', 'O75582', 'O75914', 'O76039', 'O94768', 'O94804', 'O95747', 'O95819', 'O96013', 'O96017', 'P00519', 'P00533', 'P04626', 'P04629', 'P06213', 'P06239', 'P06493', 'P07332', 'P07333', 'P07949', 'P08069', 'P08581', 'P08631', 'P08922', 'P10721', 'P11309', 'P11362', 'P11802', 'P12931', 'P15056', 'P15735', 'P16234', 'P17252', 'P17612', 'P19525', 'P19784', 'P21802', 'P21860', 'P22455', 'P22607', 'P23443', 'P23458', 'P24723', 'P24941', 'P25098', 'P27037', 'P27361', 'P27448', 'P28482', 'P29317', 'P29320', 'P29322', 'P29597', 'P30291', 'P31749', 'P31751', 'P33981', 'P34925', 'P34947', 'P35968', 'P36888', 'P36897', 'P37173', 'P41279', 'P41743', 'P42336', 'P42345', 'P42684', 'P43403', 'P43405', 'P45983', 'P45984', 'P45985', 'P48729', 'P48730', 'P48736', 'P49137', 'P49336', 'P49759', 'P49760', 'P49761', 

In [12]:
import base64
import gzip

OUTPUT_CSV_WITH_PROTEIN = OUTPUT_CSV.replace(".csv", "_with_protein_mol2.csv")

def fetch_protein_mol2(structure_id: int) -> str:
    response = session.get(
        f"{BASE_URL}/structure_get_protein",
        params={"structure_ID": int(structure_id)},
        timeout=REQUEST_TIMEOUT,
        headers={"Accept": "chemical/x-mol2, text/plain, */*"},
    )
    response.raise_for_status()
    return response.text


def compress_mol2_to_base64(mol2_text: str) -> str:
    compressed = gzip.compress(mol2_text.encode("utf-8"), compresslevel=9)
    return base64.b64encode(compressed).decode("ascii")


df_with_protein = best_per_uniprot.copy()
df_with_protein["protein_mol2_gzip_base64"] = None
df_with_protein["protein_mol2_fetch_error"] = None

for i, structure_id in enumerate(df_with_protein["structure_ID"], start=1):
    try:
        mol2_text = fetch_protein_mol2(structure_id)
        df_with_protein.at[i - 1, "protein_mol2_gzip_base64"] = compress_mol2_to_base64(mol2_text)
    except Exception as exc:
        df_with_protein.at[i - 1, "protein_mol2_fetch_error"] = str(exc)

    if i % 25 == 0 or i == len(df_with_protein):
        print(f"Processed {i}/{len(df_with_protein)} structures")

df_with_protein.to_csv(OUTPUT_CSV_WITH_PROTEIN, index=False)
print(f"Saved {len(df_with_protein):,} rows to {OUTPUT_CSV_WITH_PROTEIN}")

df_with_protein[["structure_ID", "uniprot", "protein_mol2_fetch_error"]].head(10)

Processed 25/225 structures
Processed 50/225 structures
Processed 75/225 structures
Processed 100/225 structures
Processed 125/225 structures
Processed 150/225 structures
Processed 175/225 structures
Processed 200/225 structures
Processed 225/225 structures
Saved 225 rows to klifs_filtered_structures_uniprot_with_protein_mol2.csv


,structure_ID,uniprot,protein_mol2_fetch_error
0,1500,O00238,None
1,13841,O00311,None
2,1818,O00444,None
3,463,O00506,None
4,12695,O14733,None
5,13496,O14757,None
6,13583,O14936,None
7,1977,O14965,None
8,336,O14976,None
9,6987,O15075,None
